## Lab 06

### Zadanie 1 - Regexy

Przejdź 15 zadań ze strony http://regexone.com/

### Zadanie 2 - Regexy

_Atoms_

Napisz program, który (używając wyrażeń regularnych) przefiltruje plik atoms.log tak by pozostały jedynie linie dot. bezproblemowego przebiegu eksperymentu.
czyli np. powinny zostać odrzucone takie linie jak
```
RUN 000039 COMPLETED. OUTPUT IN FILE yttrium.dat. 1 UNDERFLOW WARNING.
```
czy
```
RUN 000058 COMPLETED. OUTPUT IN FILE cerium.dat. ALGORITHM DID NOT CONVERGE AFTER 100000 ITERATIONS.
```
można jedynie założyć, że każda linijka pasuje do poniższego schematu
```
RUN [[NUMER Z DOKŁADNIE 6 CYFR]] COMPLETED. OUTPUT IN FILE [[NAZWA]].dat. [[COKOLWIEK]]
```

In [1]:
import re
from os import PathLike
from pathlib import Path

SUCCESSFUL_RUN = re.compile(
    r"^RUN \d{6} COMPLETED\. OUTPUT IN FILE [^.]+\.dat\.\s*$"
)


def iter_successful_runs(source):
    """Yield runs whose log record has no trailing diagnostic text."""
    should_close = isinstance(source, (str, PathLike))
    lines = open(source, encoding="utf-8") if should_close else source
    try:
        yield from (line for line in lines if SUCCESSFUL_RUN.fullmatch(line))
    finally:
        if should_close:
            lines.close()


atoms_log = Path("atoms.log")
if atoms_log.is_file():
    for line in iter_successful_runs(atoms_log):
        print(line, end="")
else:
    print("atoms.log missing; iter_successful_runs is ready for a supplied log file.")

atoms.log missing; iter_successful_runs is ready for a supplied log file.


### Zadanie 3 - Regexy

_Messages_

Napisz program który wypisze linie z logu systemowego "messages.txt" które zawierają informacje o nieprawidłowej nazwie użytkownika (invalid user)

In [2]:
import re
from os import PathLike

INVALID_USER = re.compile(r"\binvalid user\b", re.IGNORECASE)


def iter_invalid_user_lines(source):
    """Yield log lines that report an invalid user name."""
    should_close = isinstance(source, (str, PathLike))
    lines = open(source, encoding="utf-8") if should_close else source
    try:
        yield from (line for line in lines if INVALID_USER.search(line))
    finally:
        if should_close:
            lines.close()


messages_log = Path("messages.txt")
if messages_log.is_file():
    for line in iter_invalid_user_lines(messages_log):
        print(line, end="")
else:
    print("messages.txt missing; iter_invalid_user_lines is ready for a supplied log file.")

messages.txt missing; iter_invalid_user_lines is ready for a supplied log file.


Przerób program tak, żeby wyświetlał sformatowane komunikaty, tzn.
```
Jun 29 20:18:40 noether sshd[5805]: Invalid user tester from 218.189.194.200
```
powinno zostać przerobione na
```
Niepoprawna nazwa użytkownika: "tester", połaczenie z 218.189.194.200 nawiązano 29 czerwca o godz. 20:18
```

In [3]:
POLISH_MONTHS = {
    "Jan": "stycznia", "Feb": "lutego", "Mar": "marca", "Apr": "kwietnia",
    "May": "maja", "Jun": "czerwca", "Jul": "lipca", "Aug": "sierpnia",
    "Sep": "września", "Oct": "października", "Nov": "listopada", "Dec": "grudnia",
}
INVALID_USER_RECORD = re.compile(
    r"^(?P<month>[A-Z][a-z]{2})\s+(?P<day>\d{1,2})\s+"
    r"(?P<time>\d{2}:\d{2}):\d{2}\s+\S+\s+sshd(?:\[\d+\])?:\s+"
    r"Invalid user\s+(?P<username>\S+)\s+from\s+(?P<host>\S+)",
    re.IGNORECASE,
)


def format_invalid_user_messages(source):
    """Yield Polish messages for parseable SSH 'Invalid user' log records."""
    for line in iter_invalid_user_lines(source):
        match = INVALID_USER_RECORD.search(line)
        if match:
            data = match.groupdict()
            month = POLISH_MONTHS[data["month"].title()]
            yield (
                f'Niepoprawna nazwa użytkownika: "{data["username"]}", '
                f'połączenie z {data["host"]} nawiązano {data["day"]} {month} '
                f'o godz. {data["time"]}'
            )


if messages_log.is_file():
    for message in format_invalid_user_messages(messages_log):
        print(message)
else:
    print("Formatting skipped: messages.txt is missing.")

Formatting skipped: messages.txt is missing.


### Zadanie 4 - Collections
Uzupenij ponizsze funckje korzystajac z zaimporotwanych klas

In [4]:
from collections import defaultdict


def count_letters(word):
    """Zwraca słownik: litera -> liczba jej wystąpień."""
    counts = defaultdict(int)
    for letter in word:
        counts[letter] += 1
    return dict(counts)


print(count_letters('aaaaa') == {'a': 5})
print(count_letters('abbccaaaa') == {'a': 5, 'b': 2, 'c': 2})
print(count_letters('abc') == {'a': 1, 'b': 1, 'c': 1})

True
True
True


In [5]:
def group_words_by_first_letter(text):
    """Dzieli tekst po białych znakach i grupuje słowa po pierwszej literze."""
    groups = defaultdict(list)
    for word in text.split():
        groups[word[0]].append(word)
    return dict(groups)


print(group_words_by_first_letter("ala ma kota") == {'a': ['ala'], 'm': ['ma'], 'k': ['kota']})
print(group_words_by_first_letter("ala ma kota ala ma psa") == {'a': ['ala', 'ala'], 'm': ['ma', 'ma'], 'k': ['kota'], 'p': ['psa']})
print(group_words_by_first_letter("ala ma kota ale marysia ma konia") == {'a': ['ala', 'ale'], 'm': ['ma', 'marysia', 'ma'], 'k': ['kota', 'konia']})

True
True
True


### Zadanie 5 - Klasy
Uzupełnij brakujące metody poniższej klasy.

In [6]:
from collections.abc import Mapping
from types import MappingProxyType


class FrozenDictionary(object):
    """
    Odpowiednik frozenset dla zbiorów, czyli słownik, który nie jest modyfikowalny,
    a dzięki temu może być np. elementem zbiorów, albo kluczem w innym słowniku.
    """

    def __init__(self, dictionary):
        """Tworzy nowy zamrożony słownik z podanego słownika."""
        self._data = MappingProxyType(dict(dictionary))
        self._hash = hash(frozenset(self._data.items()))

    def __hash__(self):
        """Zwraca hasz słownika (int)."""
        return self._hash

    def __eq__(self, d):
        """Porównuje nasz słownik z innym słownikiem lub FrozenDictionary."""
        if isinstance(d, FrozenDictionary):
            return self._data == d._data
        if isinstance(d, Mapping):
            return dict(self._data) == dict(d)
        return NotImplemented

    def __repr__(self):
        """Zwraca reprezentację naszego słownika jako string."""
        return repr(dict(self._data))


dicts = [FrozenDictionary({'ala': 4}),
         FrozenDictionary({'ala': 1, 'jacek': 0}),
         FrozenDictionary({'ala': 4}),
         FrozenDictionary({'ala': 2}),
         FrozenDictionary({'jacek': 0, 'ala': 1})]

s = set(dicts)
print(dicts[0] == dicts[2])
print(dicts[0] != dicts[3])
print(len(s) == 3)
for d in dicts:
    print(d in s)

# Powinno wyświetlić coś w stylu set([{'ala': 4}, {'ala': 1, 'jacek': 0}, {'ala': 2}])
print(s)

True
True
True
True
True
True
True
True
{{'ala': 2}, {'ala': 1, 'jacek': 0}, {'ala': 4}}


### Zadanie 6 - Bag of Words
Napisz klasę "Bag of words", która będzie "workiem słów" z zadanego dokumentu. Oznacza to, że ma w sobie przechowywać informacje o tym ile razy każde słowo wystąpiło w dokumencie. Dodatkowo ma udostępniać następującą funkcjonalność:
* Można utworzyć go na dwa sposoby:
```
bow = BagOfWords("ala ma kota ala ma ala")
bow = BagOfWords(open("plik.txt"))
```
* Możliwość wyświetlania go po prostu pisząc print(bow) (kolejność taka sama jak przy przeglądaniu forem)
```
ala:3, ma:2, kota:1
```
* Sprawdzanie czy jakieś słowo należy do naszego worka pisząc
```
if 'ala' in bow:
    ...
```
* Przeglądanie słów w worku od najczęściej do najrzadziej występującego, czyli
```
for word in bow:
  print(word)
```
    powinniśmy zobaczyć
```
ala
ma
kota
```
* Możliwość dodawania dwóch worków, pisząc
```
bow1 = BagOfWords("ala ma kota ala ma ala")
bow2 = BagOfWords("tomek tez ma kota")
bow3 = bow1 + bow2
print('tomek' in bow1) # False
print('tomek' in bow3) # True
print('ala' in bow3) # True
print(bow3) # ala:3, ma:3, kota:2, tez:1, tomek:1
    ```
* Odczytywanie częstości wystąpień słów poprzez nawiasy kwadratowe
```
print(bow1["ala"]) # 3
print(bow3["ala"]) # 3
```
* Aktualizację cześtości wystąpień
```
bow3['tomek'] = 10
for el in bow3:
    print el
```
    powinno wyświetlić najpierw `tomek`

In [7]:
from collections import Counter


class BagOfWords:
    """Zlicza słowa z tekstu albo iterowalnego źródła linii tekstu."""

    def __init__(self, document):
        self._counts = Counter()
        if isinstance(document, str):
            self._counts.update(document.split())
        else:
            for chunk in document:
                self._counts.update(chunk.split())

    def __str__(self):
        return ", ".join(f"{word}:{self._counts[word]}" for word in self)

    __repr__ = __str__

    def __contains__(self, word):
        return word in self._counts

    def __iter__(self):
        return iter(sorted(self._counts, key=lambda word: (-self._counts[word], word)))

    def __add__(self, other):
        if not isinstance(other, BagOfWords):
            return NotImplemented
        result = BagOfWords(())
        result._counts = self._counts + other._counts
        return result

    def __getitem__(self, word):
        return self._counts[word]

    def __setitem__(self, word, count):
        if not isinstance(count, int) or isinstance(count, bool) or count < 0:
            raise ValueError("count must be a non-negative integer")
        if count:
            self._counts[word] = count
        else:
            self._counts.pop(word, None)


bow1 = BagOfWords("ala ma kota ala ma ala")
bow2 = BagOfWords("tomek tez ma kota")
bow3 = bow1 + bow2
assert str(bow1) == "ala:3, ma:2, kota:1"
assert str(bow3) == "ala:3, ma:3, kota:2, tez:1, tomek:1"
assert bow1["ala"] == bow3["ala"] == 3
assert "tomek" not in bow1 and "tomek" in bow3
bow3["tomek"] = 10
assert next(iter(bow3)) == "tomek"